# SABRE: Simulation of the coherent dynamics
Before studying this notebook, it is recommended to go through the following:
- sabre_3spin_analytical_cosine.ipynb
- sabre_3-fp_3spin_simulation_static.ipynb

This notebook contains a spin dynamics simulation that demonstrates coherent evolution of a SABRE system when subjected to a sinusoidal polarisation transfer field $B_0 + B_1 \cos(\omega_p t)$ where $B_0$ is the average offset, $B_1$ the amplitude, and $\omega_p = 2\pi / T$ the angular frequency corresponding to the period $T$. The period $T$ is chosen to be $10\ \mathrm{ms}$ and the other parameters such that $B_0$ fulfills the generalised level anticrossing (LAC) condition with $m=1$ and that $B_1$ maximises effective coupling.

## Details
- Spin system: Three spins (hydride protons + $\mathrm{^{19}F}$ from equatorial 3-fluoropyridine ligand)
- Only coherent dynamics are simulated (no chemical exchange or relaxation)
- Populations of all eight states in singlet-triplet Zeeman basis are followed

The simulations are performed using Spinguin (v. 0.0.1) package. In addition, we use the functionality from NumPy and Matplotlib for creating arrays and plotting, respectively.

In [ ]:
import spinguin as sg
from spinguin.core.hide_prints import HidePrints
import numpy as np
import matplotlib.pyplot as plt

### Build the required spin systems
We build the three-spin system that consists of the hydride protons and the $^{19}\mathrm{F}$ from the 3-fluoropyridine ligand. In addition, we construct spin systems for the hydride protons and the $^{19}\mathrm{F}$ nucleus separately in order to conveniently create the coupled states. Full Liouville space basis sets are constructed for each system (maximum spin order equal to the number of spins).

In [ ]:
ss_c = sg.SpinSystem(["1H", "1H", "19F"])
ss_c.basis.max_spin_order = 3
ss_c.basis.build()

ss_H2 = sg.SpinSystem(["1H", "1H"])
ss_H2.basis.max_spin_order = 2
ss_H2.basis.build()

ss_s = sg.SpinSystem(["19F"])
ss_s.basis.max_spin_order = 1
ss_s.basis.build()

### Define chemical shifts and $J$-couplings
We set the chemical shifts in ppm and the $J$-couplings in Hz.

In [ ]:
ss_c.chemical_shifts = [-23.1, -23.1, -124.2]

ss_c.J_couplings = [
    [0,     0,      0],
    [-7.67, 0,      0],
    [-0.05, 2.08,   0]
]

### Build the states for hydride protons and the $^{19}\mathrm{F}$
Next, we build the singlet, triplet-plus, triplet-zero, and triplet-minus states for the hydride protons and the alpha and beta states for the $^{19}\mathrm{F}$. These are used to construct the three-spin states in the next step.

In [ ]:
S = sg.singlet_state(ss_H2, 0, 1)
Tp = sg.triplet_plus_state(ss_H2, 0, 1)
T0 = sg.triplet_zero_state(ss_H2, 0, 1)
Tm = sg.triplet_minus_state(ss_H2, 0, 1)

a = sg.alpha_state(ss_s, 0)
b = sg.beta_state(ss_s, 0)

### Combine the states to get the detection states
We combine the previously built states using the associate() function to get all the eight states in the singlet-triplet Zeeman representation. These states are used to measure the populations of the states later. The spin maps are required to inform the function how the indices should be mapped to the combined system.

In [ ]:
spin_map_H2 = [0, 1]
spin_map_s = [2]

Sa = sg.associate(ss_H2, ss_s, ss_c, S, a, spin_map_H2, spin_map_s)
Tpb = sg.associate(ss_H2, ss_s, ss_c, Tp, b, spin_map_H2, spin_map_s)
T0a = sg.associate(ss_H2, ss_s, ss_c, T0, a, spin_map_H2, spin_map_s)

Sb = sg.associate(ss_H2, ss_s, ss_c, S, b, spin_map_H2, spin_map_s)
Tma = sg.associate(ss_H2, ss_s, ss_c, Tm, a, spin_map_H2, spin_map_s)
T0b = sg.associate(ss_H2, ss_s, ss_c, T0, b, spin_map_H2, spin_map_s)

Tpa = sg.associate(ss_H2, ss_s, ss_c, Tp, a, spin_map_H2, spin_map_s)
Tmb = sg.associate(ss_H2, ss_s, ss_c, Tm, b, spin_map_H2, spin_map_s)

detection = [Sa, Tpb, T0a, Sb, Tma, T0b, Tpa, Tmb]
detection_str = [r"$S\alpha$", r"$T_+\beta$", r"$T_0\alpha$",
                 r"$S\beta$", r"$T_-\alpha$", r"$T_0\beta$",
                 r"$T_+\alpha$", r"$T_-\beta$"]

### Create the time evolution propagator
Next, the parameters for the sinusoidal polarisation transfer field with the period $T = 10\ \mathrm{ms}$ are defined. We set the average offset $B_0 = 43.1\ \mathrm{\mu T}$ (matches the generalised LAC condition) and $B_1 = 73.2\ \mathrm{\mu T}$ (maximises the effective coupling). These values were calculated in the previous notebook. We approximate the sinusoidal magnetic field with $nterms = 100$ data points. For convenience, the time step of the propagator is chosen to be equal to one period of the oscillating field. We build the Hamiltonian, Liouvillian, and the time propagator. The time propagator corresponds now to one period of the wave.

In [ ]:
T = 10e-3
B0 = 43.1e-6
B1 = 73.2e-6

nterms = 100
dt = T / nterms

Bt = B0 + B1 * np.array([np.cos(i*2*np.pi / nterms) for i in range(nterms)])

P = np.eye(ss_c.basis.dim, ss_c.basis.dim)
with HidePrints():
    for i in range(nterms):
        sg.parameters.magnetic_field = Bt[i]
        H = sg.hamiltonian(ss_c)
        L = sg.liouvillian(H)
        P = sg.propagator(L, dt) @ P

### Time evolution
We simulate the spin system for a short time of $1\ \mathrm{s}$ just to illustrate the coherent oscillation. The populations, which are acquired using the detection states during each time step, are stored in an array for further analysis.

In [ ]:
time = 1
nsteps = int(time/T)
rho_c = sg.singlet_state(ss_c, 0, 1)
populations = np.zeros((nsteps, len(detection)))

for i in range(nsteps):

    for j in range(len(detection)):
        populations[i, j] = (detection[j].conj().T @ rho_c).trace().real * 100

    rho_c = P @ rho_c

### Plot the populations
The coherent evolution of the state populations is plotted. We notice the interchange of $\ket{S\beta}$ and $\ket{T_-\alpha}$ whereas the other states remain nearly constant. Comparison to the static-field case simulated in the previous notebook reveals that the coherent oscillation takes place at a reduced frequency, as expected by the modulation of the effective coupling.

In [ ]:
t = np.linspace(0, nsteps*T, nsteps, endpoint=False)
for i in range(len(detection)):
    plt.plot(t, np.real(populations[:, i]), label=detection_str[i])
plt.legend(loc="upper right")
plt.xlabel("Time (s)")
plt.ylabel("Population (%)")
plt.tight_layout()
plt.show()
plt.clf()

### Save the populations
Finally, the populations are saved as a .csv file for further analysis.

In [ ]:
np.savetxt(f"populations.csv", populations, delimiter=",")